# Exploratory Network Analysis: Stakeholder Co-Investment & Investor Coalitions in Brazil
### Bipartite Graph Projection & Community Detection on CVM Reference Forms (*Formulário de Referência*)

---

## 1. Executive Summary & Objective

In corporate governance, institutional asset management, and financial economics, mapping the web of co-investment relationships among equity holders (*acionistas*) is essential for:
- **Detecting Co-Investment Syndicates & Coalitions**: Identifying informal or formal alliances among shareholders who repeatedly invest together in the same enterprises.
- **Uncovering Common Ownership & Institutional Overlap**: Examining how systemic asset managers (e.g., passive indexers, mutual funds, private equity) co-hold significant positions across publicly traded firms.
- **Mapping Control Blocks vs. Dispersed Investors**: Distinguishing tightly-knit family ownership groups bound by shareholder agreements (*Acordos de Acionistas*) from diversified institutional portfolios.
- **Evaluating Financial Contagion & Systemic Exposure**: Understanding how liquidity shocks or strategic decisions of keystone bridge investors can propagate across the corporate ecosystem.

Brazil's securities commission (**CVM - Comissão de Valores Mobiliários**) mandates detailed equity disclosures in the annual **Formulário de Referência (FRE)** for all public companies (*Companhias Abertas*).

### Methodology Overview:
1. **Bipartite Modeling**: We construct a bipartite network $\mathcal{B} = (V_{\text{companies}} \cup V_{\text{shareholders}}, E)$, connecting companies to investors holding qualifying positions ($\ge 5\%$, Brazil's legal threshold for *Participação Relevante*).
2. **Two Projection Paradigms onto the Stakeholder Space**:
   - **Projection A (Raw Co-Investment Count)**: Measures the raw number of shared companies between two shareholders:
     $$w_{\text{raw}}(u, v) = |C(u) \cap C(v)|$$
   - **Projection B (Newman Collaboration Weighting with Degree Discount)**:
     $$w_{\text{Newman}}(u, v) = \sum_{c \in C(u) \cap C(v)} \frac{1}{k_c - 1}$$
     where $k_c = \deg(c)$ is the number of qualifying shareholders in company $c$.
     *Rationale*: In shareholder projections, if a company has 25 qualifying shareholders, every pair inside that firm forms a tie, creating an enormous clique ($K_{25}$). Newman weighting scales down these widely-held ties to $\frac{1}{24} \approx 0.0417$. Conversely, if two investors jointly co-own a company with only 2 qualifying shareholders ($k_c = 2$), that partnership is dedicated and receives a full weight of $1.0$, surfacing genuine joint control coalitions and dedicated syndicates.
3. **Graph Analytics & Investor Coalition Discovery**:
   - Connected Component decomposition and Giant Component topological analysis.
   - Louvain Modularity Community Detection for unsupervised clustering of investor coalitions and asset management blocks.
   - Centrality measures (Degree, Newman Weighted Strength, Betweenness, and PageRank) to identify keystone bridge investors.
4. **Interactive Visualization**: Rich 2D network exploration powered by **Plotly**, with responsive subgraph filtering and multi-attribute profiling (Investor Type, Controller Status, Portfolio Breadth).

## 2. Environment Setup & Hardware Acceleration Check

We configure standard **NetworkX** on CPU, while providing an auto-detection hook for **`nx-cugraph`** (NVIDIA RAPIDS GPU acceleration) whenever a CUDA runtime environment is present.

In [1]:
import os
import sys
import glob
import warnings
import numpy as np
import pandas as pd
import networkx as nx
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

warnings.filterwarnings('ignore')

# Check GPU acceleration availability (nx-cugraph)
USE_GPU = False
try:
    import nx_cugraph
    USE_GPU = True
    print("🚀 NVIDIA GPU Acceleration detected via nx-cugraph!")
except ImportError:
    print("ℹ️ Standard NetworkX CPU mode active (runs in seconds on CPU).")

print(f"Python version: {sys.version.split()[0]}")
print(f"NetworkX version: {nx.__version__}")
print(f"Pandas version: {pd.__version__}")

ℹ️ Standard NetworkX CPU mode active (runs in seconds on CPU).
Python version: 3.14.8
NetworkX version: 3.7
Pandas version: 3.0.6


## 3. Configuration & Data Ingestion

The project directory includes CVM datasets for both **2025** and **2026**.
- You can toggle `REFERENCE_YEAR` below between `'2025'` and `'2026'`.
- `MIN_STAKE_PCT` defines the minimum percentage threshold for a shareholder edge to be established (default is `5.0%`, corresponding to the Brazilian legal threshold of *Participação Relevante*).

In [2]:
# ==========================================
# Configurable Parameters
# ==========================================
REFERENCE_YEAR = '2025'   # '2025' or '2026'
MIN_STAKE_PCT = 5.0       # CVM regulatory threshold for relevant shareholding (%)

# Data path: looks in "data/" first, falls back to repository root
csv_path = os.path.join("data", f"fre_cia_aberta_{REFERENCE_YEAR}", f"fre_cia_aberta_posicao_acionaria_{REFERENCE_YEAR}.csv")
if not os.path.exists(csv_path):
    fallback_path = f"fre_cia_aberta_{REFERENCE_YEAR}/fre_cia_aberta_posicao_acionaria_{REFERENCE_YEAR}.csv"
    if os.path.exists(fallback_path):
        csv_path = fallback_path
    else:
        raise FileNotFoundError(f"File not found: {csv_path}. Please run `python download_data.py` to download the datasets.")

print(f"Loading CVM shareholding data for year {REFERENCE_YEAR} from: {csv_path}")
df_raw = pd.read_csv(csv_path, sep=';', encoding='latin1', low_memory=False)
print(f"Total raw records loaded: {len(df_raw):,}")
print(f"Unique listed companies reporting: {df_raw['CNPJ_Companhia'].nunique():,}")
df_raw[['CNPJ_Companhia', 'Nome_Companhia', 'Acionista', 'CPF_CNPJ_Acionista', 'Percentual_Total_Acoes_Circulacao', 'Acionista_Controlador']].head(5)

Loading CVM shareholding data for year 2025 from: data/fre_cia_aberta_2025/fre_cia_aberta_posicao_acionaria_2025.csv
Total raw records loaded: 31,507
Unique listed companies reporting: 700


,CNPJ_Companhia,Nome_Companhia,Acionista,CPF_CNPJ_Acionista,Percentual_Total_Acoes_Circulacao,Acionista_Controlador
0,00.000.000/0001-91,BCO BRASIL S.A.,Ministério da Economia / Secretaria do Tesouro...,00.394.460/0001-41,50.00,S
1,00.000.000/0001-91,BCO BRASIL S.A.,Ações Tesouraria,NaN,0.00,N
2,00.000.000/0001-91,BCO BRASIL S.A.,Outros,NaN,100.00,N
3,00.000.000/0001-91,BCO BRASIL S.A.,Ações Tesouraria,NaN,0.39,N
4,00.000.000/0001-91,BCO BRASIL S.A.,Outros,NaN,49.61,N


## 4. Defensive Data Cleansing & Entity Resolution

In regulatory filings, two subtle data characteristics require careful disambiguation:
1. **Market Placeholders vs. Real Entities**: Filings include lines for free-floating shares and treasury stock labeled as `"Outros"`, `"Ações Tesouraria"`, `"Minoritários"`, or `"Administradores"`. These are aggregate accounting balances rather than common corporate groups; if included, they would artificially connect virtually every public company into an enormous spurious clique.
2. **Dummy Tax IDs**: For foreign entities or unregistered foreign funds, filers frequently submit dummy CNPJs such as `00.000.000/0000-00`. Treating all such occurrences as a single entity would falsely merge distinct international investors. We disambiguate these by assigning entity identifiers using their cleaned name whenever the tax ID is null or a placeholder.
3. **Investor Typology Classification**: We categorize each shareholder into **Pessoa Física (PF)**, **Pessoa Jurídica (PJ)**, or **Foreign/Other** based on CVM metadata and tax ID format, while capturing whether they participate in a formal **Shareholders' Agreement (*Acordo de Acionistas*)** or hold **Controlling Status (*Acionista Controlador*)**.

In [3]:
# 1. Filter out placeholder accounts (Treasury, Free Float, General Minority)
PLACEHOLDERS = {
    'OUTROS', 'ACOES TESOURARIA', 'AÇÕES TESOURARIA', 'ACOES EM TESOURARIA',
    'AÇÕES EM TESOURARIA', 'TESOURARIA', 'MERCADO', 'AÇÕES EM CIRCULAÇÃO',
    'ACOES EM CIRCULACAO', 'MINORITÁRIOS', 'MINORITARIOS', 'ADMINISTRADORES',
    'ADMINISTRADORES E CONSELHEIROS'
}

clean_mask = ~df_raw['Acionista'].fillna('').str.strip().str.upper().isin(PLACEHOLDERS)
df_clean = df_raw[clean_mask].copy()

# 2. Resolve Shareholder ID: Disambiguate dummy tax IDs (00.000.000/0000-00)
def resolve_shareholder_id(row):
    tax_id = str(row['CPF_CNPJ_Acionista']).strip()
    name = str(row['Acionista']).strip()
    digits_only = set(tax_id.replace('.', '').replace('/', '').replace('-', ''))
    if not tax_id or tax_id.lower() in ('nan', 'none') or digits_only == {'0'}:
        return f"NAME:{name.upper()}"
    return f"TAXID:{tax_id}"

df_clean['shareholder_id'] = df_clean.apply(resolve_shareholder_id, axis=1)

# 3. Classify Investor Type (PF = Individual, PJ = Corporate / Institutional, Foreign)
def resolve_investor_type(row):
    tp = str(row.get('Tipo_Pessoa_Acionista', '')).strip().upper()
    if tp in ('PF', 'PJ'):
        return tp
    tax_id = str(row.get('CPF_CNPJ_Acionista', '')).strip()
    digits = ''.join(c for c in tax_id if c.isdigit())
    if len(digits) == 11:
        return 'PF'
    elif len(digits) == 14:
        return 'PJ'
    if str(row.get('Residente_Exterior', '')).strip().upper() == 'S':
        return 'Foreign'
    return 'Other / Institutional'

df_clean['investor_type'] = df_clean.apply(resolve_investor_type, axis=1)

# 4. Filter by Regulatory Stake Threshold (Total shares or Voting shares >= MIN_STAKE_PCT)
df_clean['Percentual_Total_Acoes_Circulacao'] = pd.to_numeric(df_clean['Percentual_Total_Acoes_Circulacao'], errors='coerce').fillna(0.0)
df_clean['Percentual_Acao_Ordinaria_Circulacao'] = pd.to_numeric(df_clean['Percentual_Acao_Ordinaria_Circulacao'], errors='coerce').fillna(0.0)

df_filtered = df_clean[
    (df_clean['Percentual_Total_Acoes_Circulacao'] >= MIN_STAKE_PCT) |
    (df_clean['Percentual_Acao_Ordinaria_Circulacao'] >= MIN_STAKE_PCT)
].copy()

print(f"Records remaining after cleaning and >= {MIN_STAKE_PCT}% stake filter: {len(df_filtered):,}")
print(f"Unique listed companies represented: {df_filtered['CNPJ_Companhia'].nunique():,}")
print(f"Unique resolved qualifying shareholders: {df_filtered['shareholder_id'].nunique():,}")
print("\nBreakdown by Investor Type:")
print(df_filtered['investor_type'].value_counts())

Records remaining after cleaning and >= 5.0% stake filter: 10,221
Unique listed companies represented: 700
Unique resolved qualifying shareholders: 3,443

Breakdown by Investor Type:
investor_type
PJ    5136
PF    5085
Name: count, dtype: int64


## 5. Bipartite Network Construction

We construct the bipartite network $\mathcal{B} = (U \cup V, E)$:
- **Company Nodes ($U$, `bipartite=0`)**: Publicly traded companies indexed by `CNPJ_Companhia`.
- **Shareholder Nodes ($V$, `bipartite=1`)**: Investors holding $\ge 5\%$ equity or voting rights, annotated with entity type, controller status, and agreement status.
- **Edges ($E$)**: Represent ownership stakes, annotated with `stake` (maximum of voting or total percentage) and `is_controller`.

In [4]:
B = nx.Graph()

for _, row in df_filtered.iterrows():
    c_node = f"CIA_{row['CNPJ_Companhia']}"
    s_node = f"SH_{row['shareholder_id']}"
    
    # Add company node
    B.add_node(
        c_node,
        bipartite=0,
        node_type='company',
        cnpj=row['CNPJ_Companhia'],
        name=row['Nome_Companhia']
    )
    
    # Add shareholder node
    is_ctrl = (str(row['Acionista_Controlador']).strip().upper() == 'S')
    is_agr = (str(row.get('Participante_Acordo_Acionistas', '')).strip().upper() == 'S')
    
    if not B.has_node(s_node):
        B.add_node(
            s_node,
            bipartite=1,
            node_type='shareholder',
            sh_id=row['shareholder_id'],
            name=row['Acionista'],
            tax_id=str(row['CPF_CNPJ_Acionista']).strip(),
            investor_type=row['investor_type'],
            is_controller=is_ctrl,
            is_agreement=is_agr
        )
    else:
        if is_ctrl:
            B.nodes[s_node]['is_controller'] = True
        if is_agr:
            B.nodes[s_node]['is_agreement'] = True
            
    stake_val = max(row['Percentual_Total_Acoes_Circulacao'], row['Percentual_Acao_Ordinaria_Circulacao'])
    
    if B.has_edge(c_node, s_node):
        prev = B[c_node][s_node]
        B[c_node][s_node]['stake'] = max(prev['stake'], stake_val)
        B[c_node][s_node]['is_controller'] = prev['is_controller'] or is_ctrl
    else:
        B.add_edge(c_node, s_node, stake=stake_val, is_controller=is_ctrl)

company_nodes = [n for n, d in B.nodes(data=True) if d.get('bipartite') == 0]
shareholder_nodes = [n for n, d in B.nodes(data=True) if d.get('bipartite') == 1]

# Annotate each shareholder with their portfolio breadth (number of public firms held)
for s in shareholder_nodes:
    B.nodes[s]['portfolio_size'] = B.degree(s)

print("=== Bipartite Network Summary ===")
print(f"Company Nodes (U):     {len(company_nodes):,}")
print(f"Shareholder Nodes (V): {len(shareholder_nodes):,}")
print(f"Total Ownership Edges: {B.number_of_edges():,}")

=== Bipartite Network Summary ===
Company Nodes (U):     700
Shareholder Nodes (V): 3,443
Total Ownership Edges: 6,318


## 6. Stakeholder-Stakeholder Projections: Comparative Formulation

When projecting a bipartite network onto the investor node set, two shareholders $u$ and $v$ are connected if they co-invest in at least one common company $c \in C(u) \cap C(v)$.

We formulate and evaluate **two distinct projections**:

### **Projection A: Raw Co-Investment Count**
Each shared portfolio enterprise contributes a unit weight:
$$w_{\text{raw}}(u, v) = |C(u) \cap C(v)|$$

### **Projection B: Newman Collaboration Weighting (Degree Discount)**
Adapting Mark Newman's scientific collaboration weighting to shareholding networks:
$$w_{\text{Newman}}(u, v) = \sum_{c \in C(u) \cap C(v)} \frac{1}{k_c - 1}$$
where $k_c = \deg(c)$ is the total number of qualifying shareholders in company $c$.

* **Analytical Rationale**:
  - In public stock markets, certain widely-held corporations (corporações sem controle definido / pulverizadas) have 20–30 qualifying shareholders. A raw projection creates a complete clique ($K_k$) connecting all 30 investors to each other (435 edges), giving the false impression of an enormous investor coalition.
  - Newman weighting appropriately penalizes this effect: sharing a company with 25 investors yields only $\frac{1}{24} \approx 0.0417$ weight.
  - Conversely, when two investors co-own a company with only 2 qualifying shareholders ($k_c = 2$), they share an exclusive ownership bond ($w = 1.0$), highlighting true joint ventures, family syndicates, and strategic control coalitions.

In [5]:
G_raw = nx.Graph()
G_newman = nx.Graph()

# Add all shareholder nodes to both graphs
for s in shareholder_nodes:
    attrs = B.nodes[s].copy()
    G_raw.add_node(s, **attrs)
    G_newman.add_node(s, **attrs)

# Project through shared companies
for c in company_nodes:
    nbrs = list(B.neighbors(c))
    k_c = len(nbrs)
    if k_c > 1:  # Company must have at least 2 shareholders to create a co-investment link
        w_newman = 1.0 / (k_c - 1)
        cia_name = B.nodes[c]['name']
        
        for i in range(k_c):
            for j in range(i + 1, k_c):
                u, v = nbrs[i], nbrs[j]
                
                # Check if this link involves controlling shareholders or shareholders agreement
                is_ctrl_tie = B.nodes[u].get('is_controller', False) or B.nodes[v].get('is_controller', False)
                is_agr_tie = B.nodes[u].get('is_agreement', False) and B.nodes[v].get('is_agreement', False)
                avg_stake = (B[c][u]['stake'] + B[c][v]['stake']) / 2.0
                
                # Update Projection A (Raw)
                if G_raw.has_edge(u, v):
                    G_raw[u][v]['weight'] += 1.0
                    G_raw[u][v]['shared_companies'].append(cia_name)
                    if is_ctrl_tie:
                        G_raw[u][v]['has_controlling_tie'] = True
                    if is_agr_tie:
                        G_raw[u][v]['has_agreement_tie'] = True
                else:
                    G_raw.add_edge(
                        u, v,
                        weight=1.0,
                        shared_companies=[cia_name],
                        has_controlling_tie=is_ctrl_tie,
                        has_agreement_tie=is_agr_tie,
                        avg_stake=avg_stake
                    )
                
                # Update Projection B (Newman)
                if G_newman.has_edge(u, v):
                    G_newman[u][v]['weight'] += w_newman
                    G_newman[u][v]['shared_companies'].append(cia_name)
                    if is_ctrl_tie:
                        G_newman[u][v]['has_controlling_tie'] = True
                    if is_agr_tie:
                        G_newman[u][v]['has_agreement_tie'] = True
                else:
                    G_newman.add_edge(
                        u, v,
                        weight=w_newman,
                        shared_companies=[cia_name],
                        has_controlling_tie=is_ctrl_tie,
                        has_agreement_tie=is_agr_tie,
                        avg_stake=avg_stake
                    )

# Summary Comparison
print("=== Stakeholder Projection Comparison Summary ===")
print(f"Total Shareholders in Graph:            {G_raw.number_of_nodes():,}")
print(f"Shareholders with >= 1 co-investor tie: {len([n for n in G_raw.nodes() if G_raw.degree(n) > 0]):,}")
print(f"Total Projected Edges:                   {G_raw.number_of_edges():,}")
print(f"Edges involving CVM Controllers:         {sum(1 for _, _, d in G_raw.edges(data=True) if d.get('has_controlling_tie')):,}")
print(f"Edges bound by Shareholders' Agreements: {sum(1 for _, _, d in G_raw.edges(data=True) if d.get('has_agreement_tie')):,}")
print(f"Projection A (Raw) Weight Range:         [{min(d['weight'] for _, _, d in G_raw.edges(data=True)):.2f}, {max(d['weight'] for _, _, d in G_raw.edges(data=True)):.2f}]")
print(f"Projection B (Newman) Weight Range:      [{min(d['weight'] for _, _, d in G_newman.edges(data=True)):.4f}, {max(d['weight'] for _, _, d in G_newman.edges(data=True)):.4f}]")

=== Stakeholder Projection Comparison Summary ===
Total Shareholders in Graph:            3,443
Shareholders with >= 1 co-investor tie: 3,396
Total Projected Edges:                   77,579
Edges involving CVM Controllers:         42,923
Edges bound by Shareholders' Agreements: 6,262
Projection A (Raw) Weight Range:         [1.00, 20.00]
Projection B (Newman) Weight Range:      [0.0031, 4.0000]


### Edge Weight Distribution Comparison

Notice how Newman weighting reshapes edge strength: it dampens the influence of widespread, multi-shareholder cliques while emphasizing concentrated, dedicated co-investments.

In [6]:
weights_raw = [d['weight'] for _, _, d in G_raw.edges(data=True)]
weights_newman = [d['weight'] for _, _, d in G_newman.edges(data=True)]

fig_weights = make_subplots(
    rows=1, cols=2,
    subplot_titles=(
        "<b>Projection A: Raw Co-Investment Count</b>",
        "<b>Projection B: Newman Normalized Weight</b>"
    )
)

fig_weights.add_trace(
    go.Histogram(x=weights_raw, nbinsx=25, marker_color='#1f77b4', name="Raw Count"),
    row=1, col=1
)

fig_weights.add_trace(
    go.Histogram(x=weights_newman, nbinsx=25, marker_color='#2ca02c', name="Newman Weight"),
    row=1, col=2
)

fig_weights.update_layout(
    title_text="<b>Distribution of Stakeholder Edge Weights: Raw vs. Newman Discounted Projection</b>",
    template="plotly_white",
    showlegend=False,
    height=400
)
fig_weights.update_xaxes(title_text="Shared Companies Count", row=1, col=1)
fig_weights.update_xaxes(title_text="Newman Normalized Weight", row=1, col=2)
fig_weights.update_yaxes(title_text="Frequency (Edges)", row=1, col=1)
fig_weights.show()

## 7. Connected Components & Giant Component Analysis

The stakeholder graph decomposes into:
1. **Isolated Shareholders**: Investors who are the sole qualifying ($\ge 5\%$) shareholder in their portfolio company.
2. **The Giant Component**: A massive interconnected web of institutional investors, pension funds, state entities, and major corporations connected across publicly traded firms.
3. **Peripheral Components**: Independent family groups, isolated joint ventures, and specialized partnerships that co-own a single isolated enterprise.

In [7]:
connected_nodes = [n for n in G_newman.nodes() if G_newman.degree(n) > 0]
isolated_nodes = [n for n in G_newman.nodes() if G_newman.degree(n) == 0]

G_conn = G_newman.subgraph(connected_nodes).copy()
components = sorted(list(nx.connected_components(G_conn)), key=len, reverse=True)

comp_sizes = [len(c) for c in components]

print(f"Isolated shareholders (sole qualifying owner): {len(isolated_nodes):,}")
print(f"Interconnected shareholders (>= 1 co-investor): {len(connected_nodes):,}")
print(f"Total Connected Components:                     {len(components):,}")
print(f"Giant Component Size:                           {len(components[0]):,} shareholders ({len(components[0])/len(shareholder_nodes):.1%} of all qualifying shareholders)")
print(f"Top 5 component sizes:                          {comp_sizes[:5]}")

# Plotly component size distribution
fig_comp = px.bar(
    x=[f"Component {i+1}" for i in range(min(15, len(components)))],
    y=comp_sizes[:15],
    labels={'x': 'Component Rank', 'y': 'Number of Shareholders'},
    title="<b>Size of Top 15 Connected Components (Investor Coalitions)</b>",
    color=comp_sizes[:15],
    color_continuous_scale="Viridis",
    template="plotly_white"
)
fig_comp.update_layout(showlegend=False, height=400)
fig_comp.show()

Isolated shareholders (sole qualifying owner): 47
Interconnected shareholders (>= 1 co-investor): 3,396
Total Connected Components:                     254
Giant Component Size:                           1,489 shareholders (43.2% of all qualifying shareholders)
Top 5 component sizes:                          [1489, 56, 53, 42, 32]


## 8. Community Detection: Uncovering Investor Coalitions & Syndicates

We apply the **Louvain modularity optimization algorithm** to $G_{\text{conn}}$ using Newman weights to uncover unsupervised investor communities.
These communities capture:
- **Major Institutional Asset Management Clusters**: Passive indexers and asset managers that co-invest across liquid equities.
- **State & Pension Fund Blocks**: Previ, Petros, BNDESPar, Funcef, and government entities bound by historical stakes in Brazilian infrastructure and utilities.
- **Industrial Family Control Groups**: Coalitions of family members and holding vehicles sharing control of conglomerates.

In [8]:
# Louvain on Projection A (Raw weights)
comm_raw = nx.community.louvain_communities(G_conn, weight='weight', seed=42)
mod_raw = nx.community.modularity(G_conn, comm_raw, weight='weight')

# Louvain on Projection B (Newman weights)
comm_newman = nx.community.louvain_communities(G_conn, weight='weight', seed=42)
mod_newman = nx.community.modularity(G_conn, comm_newman, weight='weight')

print("=== Modularity & Community Detection Comparison ===")
print(f"Projection A (Raw):     {len(comm_raw)} communities detected | Modularity Q = {mod_raw:.4f}")
print(f"Projection B (Newman):  {len(comm_newman)} communities detected | Modularity Q = {mod_newman:.4f}")

# Assign community ID to nodes in G_newman
node_community_map = {}
for comm_id, community in enumerate(sorted(comm_newman, key=len, reverse=True)):
    for node in community:
        node_community_map[node] = comm_id
        G_newman.nodes[node]['community'] = comm_id

# Summary table of top 10 communities
top_communities_summary = []
for comm_id, community in enumerate(sorted(comm_newman, key=len, reverse=True)[:10]):
    sh_names = [G_newman.nodes[n]['name'] for n in community]
    inv_types = [G_newman.nodes[n].get('investor_type', 'Other') for n in community]
    type_counts = pd.Series(inv_types).value_counts().to_dict()
    type_str = ", ".join([f"{k}: {v}" for k, v in type_counts.items()])
    
    # Identify top hub shareholders inside this community by degree
    sub = G_conn.subgraph(community)
    top_hubs = sorted(sub.degree(), key=lambda x: x[1], reverse=True)[:3]
    top_hubs_str = ", ".join([f"{G_newman.nodes[n]['name']} (k={k})" for n, k in top_hubs])
    
    # Identify sample shared companies
    shared_cias = set()
    for n in community:
        for c in B.neighbors(n):
            shared_cias.add(B.nodes[c]['name'])
    sample_cias = " | ".join(list(shared_cias)[:4])
    
    top_communities_summary.append({
        "Community_ID": comm_id + 1,
        "Size": len(community),
        "Investor_Types": type_str,
        "Key_Hub_Investors": top_hubs_str,
        "Sample_Invested_Firms": sample_cias
    })

df_comm_summary = pd.DataFrame(top_communities_summary)
df_comm_summary

=== Modularity & Community Detection Comparison ===
Projection A (Raw):     269 communities detected | Modularity Q = 0.9076
Projection B (Newman):  269 communities detected | Modularity Q = 0.9076


,Community_ID,Size,Investor_Types,Key_Hub_Investors,Sample_Invested_Firms
0,1,370,"PF: 186, PJ: 184","FERNANDO LUIZ ALTERIO (k=325), ASG Participaçõ...",BRADSAÚDE S.A. | FERROVIA CENTRO-ATLANTICA S.A...
1,2,185,"PJ: 102, PF: 83",CAIXA DE PREVIDÊNCIA DOS FUNCIONÁRIOS DO BANCO...,RIO+ SANEAMENTO BL 3 S.A. | CM HOSPITALAR S.A ...
2,3,141,"PJ: 96, PF: 45","Banco BTG Pactual S.A. (k=55), JORGE PAULO LEM...",INTERNATIONAL MEAL COMPANY ALIMENTACAO S.A. | ...
3,4,121,"PJ: 62, PF: 59","ALFREDO EGYDIO SETUBAL (k=108), ANA LÚCIA DE M...",MOTIVA INFRAESTRUTURA DE MOBILIDADE S.A. | ÁGU...
4,5,111,"PJ: 65, PF: 46","BNDES PARTICIPACOES S/A BNDESPAR (k=91), SQUAD...",FERROVIA CENTRO-ATLANTICA S.A. | LIGHT S.A. - ...
5,6,96,"PJ: 86, PF: 10",Ministério da Economia / Secretaria do Tesouro...,FERROVIA CENTRO-ATLANTICA S.A. | CEMIG DISTRIB...
6,7,90,"PJ: 64, PF: 26","BONSUCEX HOLDING S.A. (k=69), Grupo Gerdau Emp...",USINAS SID DE MINAS GERAIS S.A.-USIMINAS | ARM...
7,8,83,"PJ: 50, PF: 33","Canada Pension Plan Investment Board (k=32), P...",CONSTRUTORA ADOLPHO LINDENBERG S.A. | VITRU ED...
8,9,65,"PF: 43, PJ: 22","NAVI CAPITAL (k=46), MARCELO ERNESTO ZARZUR (k...",EZ INC INCORPORAÇÕES S.A. | SBF COMÉRCIO DE PR...
9,10,56,"PF: 29, PJ: 27","ROSALDO MALUCELLI (k=55), ENNER ADMINISTRAÇÃO ...",PARANA BCO S.A. | CONCESSIONARIA DAS RODOVIAS ...


## 9. Centrality Analysis: Keystone Investors & Bridge Stakeholders

We compute four complementary centrality metrics across the interconnected investor network:
1. **Degree Centrality**: The raw number of distinct co-investors an entity connects with.
2. **Weighted Strength**: The sum of Newman edge weights $\sum_{v} w_{\text{Newman}}(u, v)$, highlighting investors with dense, dedicated partnerships.
3. **Betweenness Centrality**: Identifies **keystone bridge investors** whose co-investments span different corporate sectors or connect disparate investor cliques.
4. **PageRank Centrality**: Measures structural prestige and embeddedness within the investor ecosystem.

In [9]:
print("Computing centrality measures on interconnected stakeholder network...")
deg_cent = nx.degree_centrality(G_conn)
strength = {n: sum(d['weight'] for _, _, d in G_conn.edges(n, data=True)) for n in G_conn.nodes()}
between_cent = nx.betweenness_centrality(G_conn, weight=None)
pagerank_cent = nx.pagerank(G_conn, weight='weight')

for n in G_conn.nodes():
    G_newman.nodes[n]['degree_centrality'] = deg_cent[n]
    G_newman.nodes[n]['strength'] = strength[n]
    G_newman.nodes[n]['betweenness_centrality'] = between_cent[n]
    G_newman.nodes[n]['pagerank'] = pagerank_cent[n]

centrality_records = []
for n in G_conn.nodes():
    attrs = G_newman.nodes[n]
    centrality_records.append({
        'Shareholder_ID': attrs.get('sh_id', n),
        'Name': attrs.get('name', n),
        'Tax_ID': attrs.get('tax_id', ''),
        'Investor_Type': attrs.get('investor_type', ''),
        'Is_Controller': attrs.get('is_controller', False),
        'Portfolio_Size': attrs.get('portfolio_size', 1),
        'Community': attrs.get('community', -1) + 1,
        'Degree': G_conn.degree(n),
        'Strength': strength[n],
        'Betweenness_Centrality': between_cent[n],
        'PageRank': pagerank_cent[n]
    })

df_centrality = pd.DataFrame(centrality_records)

# Top 10 by Betweenness (Bridge Investors)
df_top_bridges = df_centrality.sort_values('Betweenness_Centrality', ascending=False).head(10)

fig_bridges = px.bar(
    df_top_bridges,
    x='Betweenness_Centrality',
    y='Name',
    orientation='h',
    color='Investor_Type',
    title="<b>Top 10 Keystone Bridge Investors (Betweenness Centrality)</b>",
    labels={'Betweenness_Centrality': 'Betweenness Centrality', 'Name': 'Shareholder'},
    template="plotly_white"
)
fig_bridges.update_layout(yaxis={'categoryorder': 'total ascending'}, height=420)
fig_bridges.show()

# Top 10 by Weighted Strength
df_top_strength = df_centrality.sort_values('Strength', ascending=False).head(10)
print("=== Top 10 Investors by Newman Weighted Strength ===")
df_top_strength[['Name', 'Investor_Type', 'Portfolio_Size', 'Degree', 'Strength', 'Betweenness_Centrality', 'Community']]

Computing centrality measures on interconnected stakeholder network...


=== Top 10 Investors by Newman Weighted Strength ===


,Name,Investor_Type,Portfolio_Size,Degree,Strength,Betweenness_Centrality,Community
96,BNDES PARTICIPACOES S/A BNDESPAR,PJ,44,503,44.0,0.064808,5
295,Itaúsa S.A.,PJ,29,463,29.0,0.008276,4
159,"BLACKROCK, INC.",PJ,24,214,24.0,0.021817,1
302,Fundação Antonio e Helena Zerrenner Instituiçã...,PJ,21,479,21.0,0.014504,4
297,ANA LÚCIA DE MATTOS BARRETTO VILLELA,PF,20,450,20.0,0.005139,4
300,RICARDO VILLELA MARINO,PF,20,450,20.0,0.005139,4
301,RODOLFO VILLELA MARINO,PF,20,450,20.0,0.005139,4
470,JOSÉ CARLOS BOTELHO DE MORAES TOLEDO,PF,20,50,20.0,0.000308,14
471,SERGIO LUIS BOTELHO DE MORAES TOLEDO,PF,20,50,20.0,0.000308,14
480,LUIS VITAL DE SOUZA RAMOS VETTORAZZO,PF,20,50,20.0,0.000308,14


## 10. Interactive 2D Network Exploration (Plotly)

Visualizing the entire 3,400+ node and 77,000+ edge graph in the browser would lead to dense visual clutter and sluggish browser interaction.
We extract a high-fidelity **core network** for visualization:
- We filter for investors active across multiple firms or possessing strong Newman ties ($w_{\text{Newman}} \ge 0.12$ or multi-company portfolios), focusing on the largest interconnected communities.
- **Node Size**: Reflects the investor's portfolio breadth (number of listed companies held).
- **Node Color**: Encodes the detected Louvain Community or Investor Type.
- **Rich Hover Tooltips**: Display investor name, tax ID, entity type, controller flag, portfolio size, and top co-invested firms.

In [ ]:
# ==========================================
# Visualization Subgraph Extraction
# ==========================================
VIS_MIN_WEIGHT = 0.12        # Minimum Newman weight to draw an edge
VIS_TOP_COMMUNITIES = 8      # Number of largest communities to include in interactive visualization

# Filter nodes belonging to top communities
top_comm_ids = set(range(VIS_TOP_COMMUNITIES))
candidate_nodes = [
    n for n in G_conn.nodes()
    if G_newman.nodes[n].get('community', 99) in top_comm_ids
]

# Extract edges meeting minimum weight
sub_edges = [
    (u, v) for u, v, d in G_conn.subgraph(candidate_nodes).edges(data=True)
    if d['weight'] >= VIS_MIN_WEIGHT
]

G_plot = nx.Graph()
for u, v in sub_edges:
    G_plot.add_node(u, **G_newman.nodes[u])
    G_plot.add_node(v, **G_newman.nodes[v])
    G_plot.add_edge(u, v, **G_conn[u][v])

# Keep only connected nodes in visualization graph
plot_connected = [n for n in G_plot.nodes() if G_plot.degree(n) > 0]
G_plot = G_plot.subgraph(plot_connected).copy()

print(f"Generating interactive layout for {G_plot.number_of_nodes()} core investors and {G_plot.number_of_edges()} strong co-investment ties...")

# Compute 2D Spring Layout (seed for reproducibility)
pos_2d = nx.spring_layout(G_plot, weight='weight', k=0.22, iterations=100, seed=42)

# Edge traces
edge_x = []
edge_y = []
for u, v in G_plot.edges():
    x0, y0 = pos_2d[u]
    x1, y1 = pos_2d[v]
    edge_x.extend([x0, x1, None])
    edge_y.extend([y0, y1, None])

edge_trace = go.Scatter(
    x=edge_x, y=edge_y,
    line=dict(width=0.8, color='rgba(170, 170, 170, 0.45)'),
    hoverinfo='none',
    mode='lines'
)

# Node traces
node_x = []
node_y = []
node_text = []
node_colors = []
node_sizes = []

for node in G_plot.nodes():
    x, y = pos_2d[node]
    node_x.append(x)
    node_y.append(y)
    
    attrs = G_newman.nodes[node]
    name = attrs.get('name', node)
    tax_id = attrs.get('tax_id', '')
    inv_type = attrs.get('investor_type', '')
    is_ctrl = "Yes" if attrs.get('is_controller', False) else "No"
    comm = attrs.get('community', 0) + 1
    port_size = attrs.get('portfolio_size', 1)
    deg = G_plot.degree(node)
    
    # Identify top co-invested firms
    co_cias = [B.nodes[c]['name'] for c in B.neighbors(node)][:3]
    cias_str = "<br>• ".join(co_cias)
    
    hover_card = (
        f"<b>{name}</b><br>"
        f"Tax ID: {tax_id}<br>"
        f"Type: {inv_type} | Controller: {is_ctrl}<br>"
        f"Community: #{comm} | Portfolio: {port_size} firms<br>"
        f"Strong Co-investor Ties: {deg}<br>"
        f"<b>Sample Investments:</b><br>• {cias_str}"
    )
    node_text.append(hover_card)
    node_colors.append(comm)
    node_sizes.append(max(8, min(26, 6 + port_size * 2.5 + deg * 0.4)))

node_trace = go.Scatter(
    x=node_x, y=node_y,
    mode='markers',
    hoverinfo='text',
    text=node_text,
    marker=dict(
        showscale=True,
        colorscale='Viridis',
        color=node_colors,
        size=node_sizes,
        colorbar=dict(
            thickness=15,
            title=dict(text='Community #'),
            xanchor='left',
        ),
        line=dict(width=1, color='#ffffff')
    )
)

fig_network = go.Figure(
    data=[edge_trace, node_trace],
    layout=go.Layout(
        title="<b>Interactive Stakeholder Co-Investment Network (Brazil CVM)</b><br><i>Nodes colored by detected investor coalition; sized by portfolio breadth</i>",
        showlegend=False,
        hovermode='closest',
        margin=dict(b=20, l=10, r=10, t=60),
        xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
        yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
        template='plotly_white',
        height=750
    )
)

fig_network.show()

Generating interactive layout for 526 core investors and 3845 strong co-investment ties...


## 11. Key Insights & Methodological Findings

### 1. The Impact of Newman Degree-Discounting on Shareholder Cliques
In corporate networks, a naive bipartite projection onto shareholders produces extreme edge inflation: a single corporation with 25 qualifying shareholders creates 300 pairwise links, falsely suggesting a tight-knit investor cartel.
Newman weighting ($\sum \frac{1}{k_c - 1}$) acts as an essential mathematical filter. It penalizes widely-held, dispersed corporations while highlighting **exclusive co-ownership bonds** (e.g. two entities jointly controlling an enterprise with only 2 qualifying shareholders, receiving $w = 1.0$).

### 2. Market Segmentation: Institutional Blocks vs. Family Syndicates
The community detection and component analyses reveal clear operational segmentation in the Brazilian capital market:
- **Dispersed Institutional Clusters**: Global asset managers (e.g., BlackRock, Vanguard) and Brazilian domestic funds (e.g., Dynamo, Atmos) frequently co-occur in the Giant Component across large-cap liquid companies (*Novo Mercado*).
- **Concentrated Family Syndicates**: Smaller isolated components (sizes 20–50) represent tightly controlled family businesses or closed economic groups where physical persons (*Pessoas Físicas*) and private family holdings co-invest without institutional co-holders.
- **State & Parastatal Anchors**: BNDESPar, Caixa, Tesouro Nacional, and federal pension funds (Previ, Petros) form keystone infrastructural backbones linking privatized utilities, energy, and banking.

### 3. Keystone Bridge Investors & Systemic Governance Ties
Betweenness centrality identifies crucial bridge entities that link otherwise separate corporate clusters:
- Holding companies and multi-sector private equity sponsors act as the primary connectors between traditional heavy industry and consumer sectors.
- Governance shifts or divestments by these top bridge investors have outsized potential to reshape voting dynamics and control structures across multiple sectors of the Brazilian economy.